# EMFLS performance and camp cluster snapshot

Snapshot date: 2026-10-14. Run from the repository root. The notebook reads saved JSON/CSV snapshots. GA4 `totalAdRevenue` is not direct AdSense PAGE_URL revenue. Camp per-page GA4 metrics group session totals by landing page; they are not URL-only views or lifetime totals. PAGE_SCORE is from the 2026-09-06 repository snapshot.

In [ ]:
import csv, json
from collections import defaultdict
from pathlib import Path

ga = json.loads(Path('data/performance/ga4-source-latest.json').read_text(encoding='utf-8'))
summary = json.loads(Path('reports/camp-cluster-audit-2026-10-14.json').read_text(encoding='utf-8'))
with Path('reports/camp-cluster-inventory-2026-10-14.csv').open(encoding='utf-8-sig', newline='') as f:
    camp = list(csv.DictReader(f))
with Path('reports/camp-sigungu-coverage-2026-10-14.csv').open(encoding='utf-8-sig', newline='') as f:
    coverage = list(csv.DictReader(f))

In [ ]:
# Re-aggregate the recorded source/medium rows by landing page.
rollup = defaultdict(lambda: {'screenPageViews': 0.0, 'totalAdRevenue': 0.0, 'sourceMediumRows': 0})
for row in ga['rows']:
    item = rollup[row['landingPage']]
    item['screenPageViews'] += row['screenPageViews']
    item['totalAdRevenue'] += row['totalAdRevenue']
    item['sourceMediumRows'] += 1
print('API rows:', len(ga['rows']), 'landing pages:', len(rollup))
print('site totals:', ga['totals'])
print('camp totals:', ga['campSummary']['totals'])
print('top camp sources:', ga['campSummary']['bySessionSourceMedium'][:5])

# Join 29-day landing-session totals to the camp inventory after index.html normalization.
def canonical(path):
    path = path.split('?', 1)[0].split('#', 1)[0]
    return path[:-len('index.html')] if path.endswith('/index.html') else path
landing_by_path = defaultdict(lambda: {'screenPageViews': 0.0, 'totalAdRevenue': 0.0, 'sourceMediumRows': 0})
for row in ga['landingPageTotals']:
    item = landing_by_path[canonical(row['landingPage'])]
    for key in item:
        item[key] += row[key]
camp_by_path = {r['page_url'].replace('https://emfls.github.io', '', 1): r for r in camp}
matches = [(camp_by_path[path], landing_by_path[canonical(path)]) for path in camp_by_path if canonical(path) in landing_by_path]
print('camp landing pages with GA4 rows:', len(matches), 'of', len(camp))
print('camp landing-session view/revenue sums:', sum(x[1]['screenPageViews'] for x in matches), sum(x[1]['totalAdRevenue'] for x in matches))


In [ ]:
print('camp pages:', len(camp))
print('existing PAGE_SCORE snapshot:', summary['campInventory']['existingPageScore'])
print('GA4 per-page landing-session totals:', summary['campInventory']['ga4LandingSessionMetrics'])
print('inventory sample:', camp[0])
print('municipal coverage:', len(coverage), 'matched:', sum(r['status'] == 'PAGE_EXISTS' for r in coverage))
print('missing units:', [r for r in coverage if r['status'] != 'PAGE_EXISTS'])

In [ ]:
print('recent low-view counts/rates:', summary['lowViewClusters']['clusters'])
print('five-page structural similarity samples:', summary['lowViewClusters']['templateSimilarity']['clusters'])